# Interactive clearcut finder from forest masks and embeddings

This notebook synthesizes the single-year similarity notebook and the spatiotemporal similarity notebook into an interactive clearcut QA flow.

It uses:

- **Annual Sentinel-2 true-color imagery** for year-by-year visual scrubbing.
- **LCMS Land_Cover** to define forest masks before and during a candidate event year.
- **LCMS Change v2025-11** to optionally require `Tree Removal` pixels.
- **Google annual satellite embeddings** to find pixels that look like a clicked clearcut reference point in space and through time.

Default mask mode is conservative for clearcuts: **pre-event forest, but not event-year forest**. That means a candidate must start forested and then no longer map as forest in the selected event year. If “mask out forested lands” should mean something else, change `mask_mode_widget` in the control panel.


## Acceptance criteria

- Click a reference clearcut point on the map.
- Pick pre-event and event years.
- Build LCMS forest/loss masks.
- Build embedding fixed-state, change-contrast, and reference-trajectory similarity layers.
- Display candidate clearcuts interactively.
- Scrub annual imagery and LCMS selected-change overlays to identify event timing.
- Optionally vectorize candidate patches inside a drawn AOI or reference-point buffer.


## 1. Initialize Earth Engine and imports

In [1]:
from __future__ import annotations

from pathlib import Path

import ee
import geemap
import geopandas as gpd
import ipywidgets as widgets
import pandas as pd
from IPython.display import Javascript, clear_output, display
from ipyleaflet import WidgetControl

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

## 2. Analysis defaults

In [2]:
EMBEDDING_COLLECTION = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
LCMS_ASSET = "projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11"
LCMS_STUDY_AREA = "CONUS"
SENTINEL2_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"

YEARS = list(range(2017, 2025))
DEFAULT_PRE_YEAR = 2022
DEFAULT_EVENT_YEAR = 2023

SCALE_METERS = 10
PATCH_SCALE_METERS = 30
EXPORT_CRS = "EPSG:5070"

DEFAULT_REFERENCE_LON = -81.8048667
DEFAULT_REFERENCE_LAT = 29.256675

# LCMS v2025-11 Land_Cover classes. Florida-relevant tree classes are selected by default.
FOREST_CLASS_OPTIONS = {
    "Trees": 1,
    "Shrubs & Trees Mix": 3,
    "Grass/Forb/Herb & Trees Mix": 4,
    "Barren & Trees Mix": 5,
}

# LCMS v2025-11 Change classes. For clearcuts, default to Tree Removal.
CHANGE_CLASS_OPTIONS = {
    "Wind": 1,
    "Hurricane": 2,
    "Snow or Ice Transition": 3,
    "Desiccation": 4,
    "Inundation": 5,
    "Prescribed Fire": 6,
    "Wildfire": 7,
    "Mechanical Land Transformation": 8,
    "Tree Removal": 9,
    "Defoliation": 10,
    "Southern Pine Beetle": 11,
    "Insect, Disease, or Drought Stress": 12,
    "Other Loss": 13,
    "Vegetation Successional Growth": 14,
    "Stable": 15,
}
LOSS_CLASS_OPTIONS = CHANGE_CLASS_OPTIONS

MASK_MODE_OPTIONS = {
    "Pre-event forest, hide event-year forest (clearcut default)": "pre_forest_hide_event_forest",
    "Pre-event forest only": "pre_forest_only",
    "Hide event-year forest": "hide_event_forest",
    "No forest mask": "none",
}


## 3. Load Florida extent

In [3]:
def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "config").exists():
            return candidate
    raise RuntimeError("Could not find repository root from current working directory")


REPO_ROOT = find_repo_root()
extent_path = REPO_ROOT / "config" / "extent.geojson"
florida_gdf = gpd.read_file(extent_path).to_crs("EPSG:4326")
florida = ee.Geometry(florida_gdf.geometry.iloc[0].__geo_interface__)

florida_gdf[["name", "fips"]].head()

,name,fips
0,Florida,12


## 4. Earth Engine collection helpers

In [4]:
embeddings = ee.ImageCollection(EMBEDDING_COLLECTION)
lcms = ee.ImageCollection(LCMS_ASSET).filter(ee.Filter.eq("study_area", LCMS_STUDY_AREA))
sentinel2 = ee.ImageCollection(SENTINEL2_COLLECTION)


def annual_embedding(year: int) -> ee.Image:
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    image = (
        embeddings
        .filterDate(start, end)
        .filterBounds(florida)
        .mosaic()
        .clip(florida)
    )
    return image.set("year", year).set("system:time_start", start.millis())



def mask_sentinel2_clouds(image: ee.Image) -> ee.Image:
    scl = image.select("SCL")
    clear = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    return image.updateMask(clear)


def annual_sentinel2_rgb(year: int) -> ee.Image:
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    image = (
        sentinel2
        .filterDate(start, end)
        .filterBounds(florida)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
        .map(mask_sentinel2_clouds)
        .median()
        .select(["B4", "B3", "B2"])
        .clip(florida)
    )
    return image.set("year", year).set("system:time_start", start.millis())


def lcms_year(year: int) -> ee.Image:
    image = ee.Image(lcms.filter(ee.Filter.eq("year", year)).first())
    return image.clip(florida).set("year", year)


reference_embedding = annual_embedding(DEFAULT_EVENT_YEAR)
embedding_bands = reference_embedding.bandNames()

{
    "embedding_band_count": embedding_bands.size().getInfo(),
    "first_embedding_bands": embedding_bands.slice(0, 5).getInfo(),
}

{'embedding_band_count': 64,
 'first_embedding_bands': ['A00', 'A01', 'A02', 'A03', 'A04']}

## 5. Mask and similarity helpers

In [5]:
def class_mask(image: ee.Image, band_name: str, classes: tuple[int, ...], name: str) -> ee.Image:
    if not classes:
        return ee.Image.constant(0).rename(name).clip(florida).toByte()
    values = list(classes)
    mask = image.select(band_name).remap(values, [1] * len(values), 0)
    return mask.rename(name).toByte().selfMask()


def forest_analysis_mask(pre_forest: ee.Image, event_forest: ee.Image, mode: str) -> ee.Image:
    if mode == "pre_forest_hide_event_forest":
        mask = pre_forest.unmask(0).multiply(event_forest.unmask(0).eq(0))
        return mask.rename("analysis_mask").selfMask()
    if mode == "pre_forest_only":
        return pre_forest.rename("analysis_mask").selfMask()
    if mode == "hide_event_forest":
        return event_forest.unmask(0).eq(0).rename("analysis_mask").selfMask()
    if mode == "none":
        return ee.Image.constant(1).clip(florida).rename("analysis_mask").selfMask()
    raise ValueError(f"Unknown forest mask mode: {mode}")


def reference_vector(image: ee.Image, point: ee.Geometry) -> ee.Dictionary:
    sample = image.sample(
        region=point,
        scale=SCALE_METERS,
        numPixels=1,
        geometries=True,
    ).first()
    return sample.toDictionary(embedding_bands)


def constant_vector_image(vector: ee.Dictionary) -> ee.Image:
    return ee.Image.constant(vector.values(embedding_bands)).rename(embedding_bands)


def cosine_similarity(image: ee.Image, vector: ee.Dictionary, name: str = "similarity") -> ee.Image:
    vector_image = constant_vector_image(vector)
    numerator = image.multiply(vector_image).reduce(ee.Reducer.sum())
    image_norm = image.pow(2).reduce(ee.Reducer.sum()).sqrt()
    vector_norm = vector_image.pow(2).reduce(ee.Reducer.sum()).sqrt()
    return numerator.divide(image_norm.multiply(vector_norm)).rename(name)


def embedding_delta(before: ee.Image, after: ee.Image) -> ee.Image:
    return after.subtract(before).pow(2).reduce(ee.Reducer.sum()).sqrt().rename("embedding_delta")


def event_years(pre_year: int, event_year: int) -> tuple[int, ...]:
    if pre_year >= event_year:
        raise ValueError("Pre-event year must be before event year.")
    return tuple(range(pre_year, event_year + 1))

## 6. Build clearcut scoring layers

In [6]:
def trajectory_similarity(reference_point: ee.Geometry, years: tuple[int, ...]) -> dict[str, ee.Image]:
    images = []
    for year in years:
        image = annual_embedding(year)
        vector = reference_vector(image, reference_point)
        start = ee.Date.fromYMD(year, 1, 1)
        images.append(
            cosine_similarity(image, vector)
            .set("year", year)
            .set("system:time_start", start.millis())
        )

    collection = ee.ImageCollection.fromImages(images)
    stack = ee.Image.cat(*[
        image.rename(f"trajectory_similarity_{year}")
        for image, year in zip(images, years)
    ])

    return {
        "collection": collection,
        "stack": stack,
        "mean": collection.mean().rename("trajectory_mean"),
        "minimum": collection.min().rename("trajectory_minimum"),
    }


def clearcut_layers(params: dict) -> dict[str, object]:
    years = event_years(params["pre_year"], params["event_year"])
    reference_point = ee.Geometry.Point([params["lon"], params["lat"]])

    pre_lcms = lcms_year(params["pre_year"])
    event_lcms = lcms_year(params["event_year"])
    pre_forest = class_mask(pre_lcms, "Land_Cover", params["forest_classes"], "pre_event_forest")
    event_forest = class_mask(event_lcms, "Land_Cover", params["forest_classes"], "event_year_forest")
    loss_mask = class_mask(event_lcms, "Change", params["loss_classes"], "lcms_loss")
    analysis_mask = forest_analysis_mask(pre_forest, event_forest, params["mask_mode"])

    pre_embedding = annual_embedding(params["pre_year"])
    event_embedding = annual_embedding(params["event_year"])
    clearcut_vector = reference_vector(event_embedding, reference_point)

    pre_similarity = cosine_similarity(pre_embedding, clearcut_vector, "pre_clearcut_state_similarity")
    event_similarity = cosine_similarity(event_embedding, clearcut_vector, "event_clearcut_state_similarity")
    contrast = event_similarity.subtract(pre_similarity).rename("clearcut_state_contrast")
    delta = embedding_delta(pre_embedding, event_embedding)
    trajectory = trajectory_similarity(reference_point, years)

    candidate_mask = analysis_mask.unmask(0)
    if params["require_lcms_loss"]:
        candidate_mask = candidate_mask.multiply(loss_mask.unmask(0))
    candidate_mask = candidate_mask.multiply(event_similarity.gte(params["similarity_threshold"]))
    candidate_mask = candidate_mask.multiply(trajectory["mean"].gte(params["trajectory_threshold"]))
    candidate_mask = candidate_mask.multiply(contrast.gte(params["contrast_threshold"]))
    candidate_mask = candidate_mask.multiply(delta.gte(params["delta_threshold"]))
    candidate_mask = candidate_mask.rename("clearcut_candidate_mask").selfMask()

    contrast_score = contrast.unitScale(0, 0.30).clamp(0, 1)
    delta_score = delta.unitScale(0, 0.80).clamp(0, 1)
    clearcut_score = (
        event_similarity
        .add(trajectory["mean"])
        .add(contrast_score)
        .add(delta_score)
        .divide(4)
        .rename("clearcut_score")
    )

    return {
        "reference_point": reference_point,
        "years": years,
        "pre_forest": pre_forest,
        "event_forest": event_forest,
        "loss_mask": loss_mask,
        "analysis_mask": analysis_mask,
        "pre_similarity": pre_similarity.updateMask(analysis_mask),
        "event_similarity": event_similarity.updateMask(analysis_mask),
        "contrast": contrast.updateMask(analysis_mask),
        "delta": delta.updateMask(analysis_mask),
        "trajectory_mean": trajectory["mean"].updateMask(analysis_mask),
        "trajectory_minimum": trajectory["minimum"].updateMask(analysis_mask),
        "trajectory_stack": trajectory["stack"].updateMask(analysis_mask),
        "candidate_mask": candidate_mask,
        "clearcut_score": clearcut_score.updateMask(candidate_mask),
    }

## 7. Interactive map

In [7]:
similarity_vis = {
    "min": 0.55,
    "max": 1.0,
    "palette": ["black", "blue", "cyan", "yellow", "red"],
}
contrast_vis = {
    "min": 0.0,
    "max": 0.25,
    "palette": ["white", "yellow", "orange", "red"],
}
delta_vis = {
    "min": 0.0,
    "max": 0.8,
    "palette": ["white", "purple", "red"],
}
mask_vis = {"min": 0, "max": 1, "palette": ["1b7837"]}
loss_vis = {"min": 0, "max": 1, "palette": ["ff7f00"]}
score_vis = {
    "min": 0.6,
    "max": 1.0,
    "palette": ["black", "purple", "magenta", "yellow"],
}
sentinel2_vis = {"bands": ["B4", "B3", "B2"], "min": 200, "max": 3500, "gamma": 1.2}

CLEARCUT_LAYER_NAMES = {
    "Reference point",
    "Pre-event forest mask",
    "Event-year forest mask",
    "LCMS loss mask",
    "Analysis mask",
    "Event clearcut-state similarity",
    "Clearcut-state contrast",
    "Embedding delta",
    "Trajectory mean similarity",
    "Trajectory minimum similarity",
    "Clearcut candidate score",
    "Candidate patches",
}
SCRUB_LAYER_NAMES = {
    "Annual Sentinel-2 imagery",
    "LCMS loss in scrub year",
}

current_layers: dict[str, object] = {}


def remove_layers_by_name(map_widget: geemap.Map, layer_names: set[str]) -> None:
    for layer in list(map_widget.layers):
        if getattr(layer, "name", None) in layer_names:
            map_widget.remove_layer(layer)


def remove_clearcut_layers(map_widget: geemap.Map) -> None:
    remove_layers_by_name(map_widget, CLEARCUT_LAYER_NAMES)


def remove_scrub_layers(map_widget: geemap.Map) -> None:
    remove_layers_by_name(map_widget, SCRUB_LAYER_NAMES)


def read_params() -> dict:
    return {
        "lon": float(lon_widget.value),
        "lat": float(lat_widget.value),
        "pre_year": int(pre_year_widget.value),
        "event_year": int(event_year_widget.value),
        "forest_classes": tuple(int(value) for value in forest_classes_widget.value),
        "loss_classes": tuple(int(value) for value in loss_classes_widget.value),
        "mask_mode": mask_mode_widget.value,
        "require_lcms_loss": bool(require_lcms_loss_widget.value),
        "similarity_threshold": float(similarity_threshold_widget.value),
        "trajectory_threshold": float(trajectory_threshold_widget.value),
        "contrast_threshold": float(contrast_threshold_widget.value),
        "delta_threshold": float(delta_threshold_widget.value),
    }


def add_clearcut_layers(map_widget: geemap.Map, layers: dict[str, object]) -> None:
    map_widget.addLayer(layers["pre_forest"], mask_vis, "Pre-event forest mask", False, 0.35)
    map_widget.addLayer(layers["event_forest"], mask_vis, "Event-year forest mask", False, 0.35)
    map_widget.addLayer(layers["loss_mask"], loss_vis, "LCMS loss mask", False, 0.50)
    map_widget.addLayer(layers["analysis_mask"], mask_vis, "Analysis mask", False, 0.30)
    map_widget.addLayer(layers["event_similarity"], similarity_vis, "Event clearcut-state similarity", True, 0.75)
    map_widget.addLayer(layers["contrast"], contrast_vis, "Clearcut-state contrast", False, 0.75)
    map_widget.addLayer(layers["delta"], delta_vis, "Embedding delta", False, 0.75)
    map_widget.addLayer(layers["trajectory_mean"], similarity_vis, "Trajectory mean similarity", False, 0.75)
    map_widget.addLayer(layers["trajectory_minimum"], similarity_vis, "Trajectory minimum similarity", False, 0.75)
    map_widget.addLayer(layers["clearcut_score"], score_vis, "Clearcut candidate score", True, 0.85)
    map_widget.addLayer(layers["reference_point"], {"color": "white"}, "Reference point")


In [8]:
PANEL_WIDTH_PX = 340
PANEL_BODY_HEIGHT = "66vh"

lon_widget = widgets.FloatText(value=DEFAULT_REFERENCE_LON, description="Lon", layout=widgets.Layout(width="160px"))
lat_widget = widgets.FloatText(value=DEFAULT_REFERENCE_LAT, description="Lat", layout=widgets.Layout(width="160px"))
pre_year_widget = widgets.IntSlider(value=DEFAULT_PRE_YEAR, min=min(YEARS), max=max(YEARS) - 1, description="Pre year")
event_year_widget = widgets.IntSlider(value=DEFAULT_EVENT_YEAR, min=min(YEARS) + 1, max=max(YEARS), description="Event year")
imagery_year_widget = widgets.IntSlider(
    value=DEFAULT_EVENT_YEAR,
    min=min(YEARS),
    max=max(YEARS),
    step=1,
    description="Imagery year",
    continuous_update=False,
)
imagery_play_widget = widgets.Play(value=DEFAULT_EVENT_YEAR, min=min(YEARS), max=max(YEARS), step=1, interval=900)
imagery_link = widgets.jslink((imagery_play_widget, "value"), (imagery_year_widget, "value"))
show_sentinel2_widget = widgets.Checkbox(value=True, description="Show Sentinel-2")
show_scrub_loss_widget = widgets.Checkbox(value=True, description="Show LCMS selected change")
set_event_year_button = widgets.Button(description="Use scrub year as event")
forest_classes_widget = widgets.SelectMultiple(
    options=FOREST_CLASS_OPTIONS,
    value=(FOREST_CLASS_OPTIONS["Trees"],),
    description="Forest LC",
    rows=4,
    layout=widgets.Layout(width="315px"),
)
loss_classes_widget = widgets.SelectMultiple(
    options=LOSS_CLASS_OPTIONS,
    value=(LOSS_CLASS_OPTIONS["Tree Removal"],),
    description="Change",
    rows=6,
    layout=widgets.Layout(width="315px"),
)
mask_mode_widget = widgets.Dropdown(
    options=MASK_MODE_OPTIONS,
    value="pre_forest_hide_event_forest",
    description="Mask",
    layout=widgets.Layout(width="315px"),
)
require_lcms_loss_widget = widgets.Checkbox(value=True, description="Require LCMS Tree Removal")
similarity_threshold_widget = widgets.FloatSlider(value=0.85, min=0.50, max=0.99, step=0.01, description="Similarity")
trajectory_threshold_widget = widgets.FloatSlider(value=0.80, min=0.50, max=0.99, step=0.01, description="Trajectory")
contrast_threshold_widget = widgets.FloatSlider(value=0.05, min=-0.20, max=0.40, step=0.01, description="Contrast")
delta_threshold_widget = widgets.FloatSlider(value=0.00, min=0.00, max=1.50, step=0.01, description="Delta")
search_radius_widget = widgets.FloatSlider(value=3.0, min=0.5, max=25.0, step=0.5, description="Patch km")
min_patch_pixels_widget = widgets.IntSlider(value=6, min=1, max=100, step=1, description="Min pixels")
update_button = widgets.Button(description="Update layers", button_style="primary")
patch_button = widgets.Button(description="Vectorize patches")
status_output = widgets.Output(layout={"border": "1px solid #ddd", "max_height": "120px", "overflow_y": "auto"})


def panel_section(children: list[widgets.Widget]) -> widgets.VBox:
    return widgets.VBox(children, layout=widgets.Layout(width="320px", gap="4px", padding="4px 0"))


location_section = panel_section([
    widgets.HTML("Click map to set reference point."),
    widgets.HBox([lon_widget, lat_widget]),
])

imagery_section = panel_section([
    widgets.HTML("Scrub annual imagery to time event."),
    widgets.HBox([imagery_play_widget, imagery_year_widget]),
    widgets.HBox([show_sentinel2_widget, show_scrub_loss_widget]),
    set_event_year_button,
])

event_section = panel_section([
    pre_year_widget,
    event_year_widget,
    mask_mode_widget,
    require_lcms_loss_widget,
])

classes_section = panel_section([
    forest_classes_widget,
    loss_classes_widget,
])

threshold_section = panel_section([
    similarity_threshold_widget,
    trajectory_threshold_widget,
    contrast_threshold_widget,
    delta_threshold_widget,
])

patch_section = panel_section([
    widgets.HBox([update_button, patch_button]),
    search_radius_widget,
    min_patch_pixels_widget,
])

status_section = panel_section([status_output])

panel_accordion = widgets.Accordion(
    children=[
        location_section,
        imagery_section,
        event_section,
        classes_section,
        threshold_section,
        patch_section,
        status_section,
    ],
    selected_index=1,
    layout=widgets.Layout(width="330px", max_height=PANEL_BODY_HEIGHT, overflow_y="auto"),
)
for index, title in enumerate([
    "Reference point",
    "Imagery scrubber",
    "Event/mask",
    "LCMS classes",
    "Thresholds",
    "Patch tools",
    "Status",
]):
    panel_accordion.set_title(index, title)

panel_header = widgets.HTML(
    value=(
        "<div class='artemis-panel-drag-handle'>"
        "<b>Clearcut controls</b>"
        "<span class='artemis-panel-hint'>drag header · open sections</span>"
        "</div>"
    ),
    layout=widgets.Layout(width="330px"),
)

control_panel = widgets.VBox(
    [panel_header, panel_accordion],
    layout=widgets.Layout(
        width=f"{PANEL_WIDTH_PX}px",
        max_height="78vh",
        overflow="hidden",
        border="1px solid rgba(0,0,0,0.25)",
    ),
)
control_panel.add_class("artemis-clearcut-panel")

DRAGGABLE_PANEL_JS = """
(() => {
  const styleId = "artemis-clearcut-panel-style";
  if (!document.getElementById(styleId)) {
    const style = document.createElement("style");
    style.id = styleId;
    style.textContent = `
      .artemis-clearcut-panel {
        background: rgba(255,255,255,0.96);
        border-radius: 8px;
        box-shadow: 0 8px 24px rgba(0,0,0,0.25);
        font-size: 12px;
      }
      .artemis-panel-drag-handle {
        cursor: move;
        user-select: none;
        padding: 8px 10px;
        color: #fff;
        background: #263238;
        border-radius: 7px 7px 0 0;
        display: flex;
        justify-content: space-between;
        gap: 8px;
        align-items: center;
      }
      .artemis-panel-hint {
        opacity: 0.72;
        font-size: 11px;
        font-weight: 400;
      }
      .artemis-clearcut-panel .widget-label {
        font-size: 11px;
      }
    `;
    document.head.appendChild(style);
  }

  function installDraggablePanel() {
    document.querySelectorAll(".artemis-clearcut-panel").forEach((panel) => {
      const handle = panel.querySelector(".artemis-panel-drag-handle");
      const control = panel.closest(".leaflet-control");
      if (!handle || !control || control.dataset.artemisDraggable === "1") return;
      control.dataset.artemisDraggable = "1";
      control.style.zIndex = 1000;

      if (window.L && window.L.DomEvent) {
        window.L.DomEvent.disableClickPropagation(control);
        window.L.DomEvent.disableScrollPropagation(control);
      }

      const startDrag = (event) => {
        const pointer = event.touches ? event.touches[0] : event;
        const rect = control.getBoundingClientRect();
        const offsetX = pointer.clientX - rect.left;
        const offsetY = pointer.clientY - rect.top;

        control.style.position = "fixed";
        control.style.left = `${rect.left}px`;
        control.style.top = `${rect.top}px`;
        control.style.right = "auto";
        control.style.bottom = "auto";
        control.style.margin = "0";

        const move = (moveEvent) => {
          const movePointer = moveEvent.touches ? moveEvent.touches[0] : moveEvent;
          control.style.left = `${Math.max(0, movePointer.clientX - offsetX)}px`;
          control.style.top = `${Math.max(0, movePointer.clientY - offsetY)}px`;
          moveEvent.preventDefault();
          moveEvent.stopPropagation();
        };

        const stop = () => {
          document.removeEventListener("mousemove", move);
          document.removeEventListener("mouseup", stop);
          document.removeEventListener("touchmove", move);
          document.removeEventListener("touchend", stop);
        };

        document.addEventListener("mousemove", move);
        document.addEventListener("mouseup", stop);
        document.addEventListener("touchmove", move, {passive: false});
        document.addEventListener("touchend", stop);
        event.preventDefault();
        event.stopPropagation();
      };

      handle.addEventListener("mousedown", startDrag);
      handle.addEventListener("touchstart", startDrag, {passive: false});
    });
  }

  const observer = new MutationObserver(installDraggablePanel);
  observer.observe(document.body, {childList: true, subtree: true});
  setTimeout(installDraggablePanel, 250);
  setTimeout(installDraggablePanel, 1000);
})();
"""

display(Javascript(DRAGGABLE_PANEL_JS))

m = geemap.Map(center=[DEFAULT_REFERENCE_LAT, DEFAULT_REFERENCE_LON], zoom=12)
m.add_basemap("SATELLITE")
m.addLayer(florida, {"color": "white"}, "Florida boundary", False)
m.add_control(WidgetControl(widget=control_panel, position="topright", max_width=PANEL_WIDTH_PX, max_height=650))


def handle_map_click(**kwargs) -> None:
    if kwargs.get("type") != "click":
        return
    coordinates = kwargs.get("coordinates")
    if not coordinates:
        return
    lat_widget.value = float(coordinates[0])
    lon_widget.value = float(coordinates[1])
    with status_output:
        clear_output()
        print("Reference point updated from map click. Press Update layers.")


def update_imagery_layers(_=None) -> None:
    year = int(imagery_year_widget.value)
    remove_scrub_layers(m)
    if show_sentinel2_widget.value:
        m.addLayer(annual_sentinel2_rgb(year), sentinel2_vis, "Annual Sentinel-2 imagery", True, 1.0)
    if show_scrub_loss_widget.value:
        loss = class_mask(lcms_year(year), "Change", tuple(int(value) for value in loss_classes_widget.value), "scrub_year_loss")
        m.addLayer(loss, loss_vis, "LCMS loss in scrub year", True, 0.55)

    # Keep analytical overlays above opaque imagery tiles.
    if current_layers:
        remove_clearcut_layers(m)
        add_clearcut_layers(m, current_layers)

    with status_output:
        clear_output()
        print(f"Showing annual imagery/change for {year}.")
        print("Use Play or Imagery year slider to find first post-harvest year; orange overlay is selected LCMS Change class.")


def set_event_year_from_scrubber(_=None) -> None:
    event_year = int(imagery_year_widget.value)
    event_year_widget.value = event_year
    if pre_year_widget.value >= event_year:
        pre_year_widget.value = max(min(YEARS), event_year - 1)
    with status_output:
        clear_output()
        print(f"Event year set to {event_year}; pre year is {pre_year_widget.value}.")
        print("Press Update layers to rebuild clearcut candidates.")


def update_layers(_=None) -> None:
    global current_layers
    params = read_params()
    with status_output:
        clear_output()
        print("Building Earth Engine layers...")
        print(params)
    try:
        layers = clearcut_layers(params)
    except Exception as exc:
        with status_output:
            clear_output()
            print(f"Could not build layers: {exc}")
        return

    remove_clearcut_layers(m)
    add_clearcut_layers(m, layers)
    m.centerObject(layers["reference_point"], 13)
    current_layers = layers
    with status_output:
        clear_output()
        print(f"Updated clearcut layers for {params['pre_year']} → {params['event_year']}.")
        print("Scrub imagery to verify event timing; use layer control to toggle products.")


def current_patch_region() -> ee.Geometry:
    user_roi = getattr(m, "user_roi", None)
    if user_roi is not None:
        return user_roi
    point = ee.Geometry.Point([float(lon_widget.value), float(lat_widget.value)])
    return point.buffer(float(search_radius_widget.value) * 1000).bounds()


def build_candidate_patches(candidate_mask: ee.Image, region: ee.Geometry, min_pixels: int) -> ee.FeatureCollection:
    integer_mask = candidate_mask.mask().selfMask().toByte().rename("candidate")
    connected = integer_mask.connectedPixelCount(100, True)
    patch_mask = integer_mask.updateMask(connected.gte(min_pixels))
    patches = patch_mask.reduceToVectors(
        geometry=region,
        scale=PATCH_SCALE_METERS,
        geometryType="polygon",
        eightConnected=True,
        labelProperty="clearcut_candidate",
        reducer=ee.Reducer.countEvery(),
        maxPixels=1e9,
        tileScale=4,
    )

    def add_area(feature: ee.Feature) -> ee.Feature:
        area_ha = feature.geometry().area(1).divide(10_000)
        return feature.set("area_ha", area_ha)

    return patches.map(add_area)


def vectorize_patches(_=None) -> None:
    if not current_layers:
        update_layers()
    if not current_layers:
        return
    region = current_patch_region()
    patches = build_candidate_patches(
        ee.Image(current_layers["candidate_mask"]),
        region,
        int(min_patch_pixels_widget.value),
    )
    remove_clearcut_layers(m)
    add_clearcut_layers(m, current_layers)
    m.addLayer(patches, {"color": "yellow"}, "Candidate patches")
    with status_output:
        clear_output()
        print("Candidate patches added. Draw an AOI first to limit vectorization, or use Patch km buffer.")


m.on_interaction(handle_map_click)
imagery_year_widget.observe(update_imagery_layers, names="value")
show_sentinel2_widget.observe(update_imagery_layers, names="value")
show_scrub_loss_widget.observe(update_imagery_layers, names="value")
set_event_year_button.on_click(set_event_year_from_scrubber)
update_button.on_click(update_layers)
patch_button.on_click(vectorize_patches)

update_imagery_layers()
update_layers()
m


<IPython.core.display.Javascript object>

Map(center=[29.256675, -81.8048667], controls=(WidgetControl(options=['position', 'transparent_bg'], position=…

## 8. Point diagnostics

In [9]:
def diagnostics_at_reference(layers: dict[str, object]) -> pd.DataFrame:
    diagnostic_image = ee.Image.cat([
        ee.Image(layers["pre_similarity"]),
        ee.Image(layers["event_similarity"]),
        ee.Image(layers["contrast"]),
        ee.Image(layers["delta"]),
        ee.Image(layers["trajectory_mean"]),
        ee.Image(layers["trajectory_minimum"]),
        ee.Image(layers["candidate_mask"]),
        ee.Image(layers["clearcut_score"]),
    ])
    sample = diagnostic_image.sample(
        region=ee.Geometry(layers["reference_point"]),
        scale=SCALE_METERS,
        numPixels=1,
        geometries=True,
    ).first()
    return geemap.ee_to_df(ee.FeatureCollection([sample]), remove_geom=False).drop(columns=["geo"], errors="ignore")


if current_layers:
    diagnostics_at_reference(current_layers)
else:
    print("Run the interactive map cell first.")

## 9. Annual fixed-state similarity table

In [10]:
def fixed_state_similarity_stack(reference_point: ee.Geometry, reference_year: int, years: list[int]) -> ee.Image:
    reference_image = annual_embedding(reference_year)
    vector = reference_vector(reference_image, reference_point)
    images = [
        cosine_similarity(annual_embedding(year), vector, f"clearcut_state_similarity_{year}")
        for year in years
    ]
    return ee.Image.cat(*images)


def fixed_state_time_series(layers: dict[str, object], years: list[int]) -> pd.DataFrame:
    params = read_params()
    stack = fixed_state_similarity_stack(
        ee.Geometry(layers["reference_point"]),
        params["event_year"],
        years,
    )
    sample = stack.sample(
        region=ee.Geometry(layers["reference_point"]),
        scale=SCALE_METERS,
        numPixels=1,
        geometries=True,
    ).first()
    df = geemap.ee_to_df(ee.FeatureCollection([sample]), remove_geom=False).drop(columns=["geo"], errors="ignore")
    columns = [f"clearcut_state_similarity_{year}" for year in years]
    return df[columns].T.rename(columns={0: "reference_point"}).rename_axis("year")


if current_layers:
    fixed_state_time_series(current_layers, YEARS)
else:
    print("Run the interactive map cell first.")

## 10. Temporal event evidence table

In [11]:
def annual_reference_evidence(layers: dict[str, object], years: list[int]) -> pd.DataFrame:
    params = read_params()
    point = ee.Geometry(layers["reference_point"])
    event_vector = reference_vector(annual_embedding(params["event_year"]), point)
    forest_classes = tuple(int(value) for value in forest_classes_widget.value)
    loss_classes = tuple(int(value) for value in loss_classes_widget.value)

    features = []
    for year in years:
        year_lcms = lcms_year(year)
        image = ee.Image.cat([
            cosine_similarity(annual_embedding(year), event_vector, "clearcut_state_similarity"),
            class_mask(year_lcms, "Land_Cover", forest_classes, "is_forest").unmask(0),
            class_mask(year_lcms, "Change", loss_classes, "is_loss").unmask(0),
            year_lcms.select("Land_Cover").rename("land_cover"),
            year_lcms.select("Change").rename("change"),
        ])
        sample = image.sample(region=point, scale=SCALE_METERS, numPixels=1, geometries=True).first()
        features.append(ee.Feature(sample).set("year", year))

    df = geemap.ee_to_df(ee.FeatureCollection(features), remove_geom=False).drop(columns=["geo"], errors="ignore")
    columns = ["year", "clearcut_state_similarity", "is_forest", "is_loss", "land_cover", "change"]
    return df[columns].sort_values("year")


if current_layers:
    annual_reference_evidence(current_layers, YEARS)
else:
    print("Run the interactive map cell first.")

EEException: User memory limit exceeded.

## 11. Optional export

In [ ]:
# Uncomment after updating layers. This exports the active candidate rasters to Google Drive.
# params = read_params()
# export_image = ee.Image.cat([
#     ee.Image(current_layers["analysis_mask"]),
#     ee.Image(current_layers["loss_mask"]),
#     ee.Image(current_layers["event_similarity"]),
#     ee.Image(current_layers["contrast"]),
#     ee.Image(current_layers["delta"]),
#     ee.Image(current_layers["trajectory_mean"]),
#     ee.Image(current_layers["trajectory_minimum"]),
#     ee.Image(current_layers["candidate_mask"]),
#     ee.Image(current_layers["clearcut_score"]),
# ])
#
# task = ee.batch.Export.image.toDrive(
#     image=export_image,
#     description=f"interactive_clearcut_candidates_{params['pre_year']}_{params['event_year']}",
#     folder="artemis_exports",
#     fileNamePrefix=f"interactive_clearcut_candidates_{params['pre_year']}_{params['event_year']}",
#     region=florida,
#     scale=PATCH_SCALE_METERS,
#     crs=EXPORT_CRS,
#     maxPixels=1e13,
# )
#
# task.start()
# task.status()

## Interpretation notes

- **Event clearcut-state similarity**: pixels that look like the clicked reference in the event year.
- **Clearcut-state contrast**: event-year similarity minus pre-event similarity. High values mean the pixel became more like the clicked clearcut state.
- **Embedding delta**: magnitude of embedding change between pre-event and event years.
- **Trajectory mean/minimum similarity**: places following the reference point across the selected year window.
- **Clearcut candidate score**: displayed only where forest/change/threshold rules pass. Tighten thresholds to reduce false positives; loosen them when known clearcuts are missing.
- **Imagery scrubber**: use Play or the year slider to find the first year where canopy disappears; then click `Use scrub year as event` and rebuild candidates.
